In [ ]:
import pandas as pd
import numpy as np

# Load datasets
cust_df = pd.read_csv("FinTrust_Customer_Data - FinTrust_Customer_Data.csv")
trans_df = pd.read_csv("FinTrust_Transaction_Data - FinTrust_Transaction_Data.csv")

# 1. Basic profiling
print("--- CUSTOMER DATASET PROFILING ---")
print(f"Records: {len(cust_df)}, Columns: {len(cust_df.columns)}")
print("Duplicates:", cust_df.duplicated().sum())
print("Missing values:\n", cust_df.isnull().sum()[cust_df.isnull().sum() > 0])

print("\n--- TRANSACTION DATASET PROFILING ---")
print(f"Records: {len(trans_df)}, Columns: {len(trans_df.columns)}")
print("Duplicates:", trans_df.duplicated().sum())
print("Missing values:\n", trans_df.isnull().sum()[trans_df.isnull().sum() > 0])

# Check data types and unusual values
print("\nTransaction Amount Stats:")
print(trans_df['Amount_NGN'].describe())

# Check referral integrity
orphan_tx = set(trans_df['Customer_ID']) - set(cust_df['Customer_ID'])
print("Orphan Transactions (Customer_ID not in Customer table):", len(orphan_tx))

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Merge datasets
merged_df = trans_df.merge(cust_df, on='Customer_ID', how='left')

# Fig 1: Volume & Count by Channel
fig, ax1 = plt.subplots(figsize=(10, 5))
channel_agg = merged_df.groupby('Channel').agg({'Amount_NGN': 'sum', 'Transaction_ID': 'count'}).reset_index()
channel_agg['Amount_M_NGN'] = channel_agg['Amount_NGN'] / 1e6
channel_agg = channel_agg.sort_values(by='Amount_M_NGN', ascending=False)

sns.barplot(data=channel_agg, x='Channel', y='Amount_M_NGN', palette='Blues_r', ax=ax1)
ax1.set_ylabel('Volume Total (Millions NGN)', fontweight='bold', color='#0F2043')
ax1.set_title('Visualisation 1: Volume des Transactions par Canal (Millions NGN)', fontsize=14, fontweight='bold', color='#0F2043')
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.1f}M", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.savefig("py_vis1_channel_volume.png", dpi=200)
plt.close()

# Fig 2: Risk Flag Rate by Transaction Type
fig, ax2 = plt.subplots(figsize=(10, 5))
tx_risk = merged_df.groupby('Transaction_Type').agg({'Risk_Review_Flag': lambda x: (x == 'Yes').mean() * 100}).reset_index()
tx_risk = tx_risk.sort_values(by='Risk_Review_Flag', ascending=False)

sns.barplot(data=tx_risk, x='Transaction_Type', y='Risk_Review_Flag', palette='Reds_r', ax=ax2)
ax2.set_ylabel('Taux de Révision de Risque (%)', fontweight='bold', color='#0F2043')
ax2.set_title('Visualisation 2: Taux de Révision de Risque par Type de Transaction', fontsize=14, fontweight='bold', color='#0F2043')
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.savefig("py_vis2_risk_by_txtype.png", dpi=200)
plt.close()

# Fig 3: International Transaction vs Risk Flag
fig, ax3 = plt.subplots(figsize=(8, 5))
intl_risk = merged_df.groupby('International_Transaction').agg({'Risk_Review_Flag': lambda x: (x == 'Yes').mean() * 100}).reset_index()

sns.barplot(data=intl_risk, x='International_Transaction', y='Risk_Review_Flag', palette=['#3896DE', '#D9534F'], ax=ax3)
ax3.set_ylabel('Taux de Risque (%)', fontweight='bold', color='#0F2043')
ax3.set_xlabel('Transaction Internationale ?', fontweight='bold')
ax3.set_title('Visualisation 3: Taux de Risque : Locale vs Internationale', fontsize=14, fontweight='bold', color='#0F2043')
for p in ax3.patches:
    ax3.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.savefig("py_vis3_intl_risk.png", dpi=200)
plt.close()

# Fig 4: Transaction Status by Channel
fig, ax4 = plt.subplots(figsize=(10, 5))
status_channel = pd.crosstab(merged_df['Channel'], merged_df['Transaction_Status'], normalize='index') * 100
status_channel.plot(kind='bar', stacked=True, colormap='Blues', ax=ax4)
ax4.set_ylabel('Proportion (%)', fontweight='bold', color='#0F2043')
ax4.set_title('Visualisation 4: Distribution du Statut des Transactions par Canal', fontsize=14, fontweight='bold', color='#0F2043')
ax4.legend(title='Statut', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("py_vis4_status_by_channel.png", dpi=200)
plt.close()

# Fig 5: Income Band vs Digital Engagement
fig, ax5 = plt.subplots(figsize=(10, 5))
income_order = ['Below 100k', '100k-249k', '250k-499k', '500k-999k', '1m+']
sns.boxplot(data=cust_df, x='Monthly_Income_Band', y='Digital_Engagement_Score', order=income_order, palette='Blues', ax=ax5)
ax5.set_ylabel('Score d\'Engagement Digital', fontweight='bold', color='#0F2043')
ax5.set_xlabel('Tranche de Revenu Mensuel', fontweight='bold')
ax5.set_title('Visualisation 5: Score d\'Engagement Digital par Tranche de Revenu', fontsize=14, fontweight='bold', color='#0F2043')
plt.tight_layout()
plt.savefig("py_vis5_income_vs_engagement.png", dpi=200)
plt.close()

print("All 5 Python EDA Visualizations generated successfully.")

: 